In [8]:
import anndata
import numpy as np
import pandas as pd
import os

# --- 1. 配置路径 ---
h5ad_dir = r"C:\Users\28616\Desktop\spatialDE_rawdata"
cluster_dir = r"C:\Users\28616\Desktop\spCLUE_Results"
save_dir = r"C:\Users\28616\Desktop\gene_cluster_expression"

# 如果文件夹不存在则创建
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

sample_ids = ["49", "50", "51", "52", "53", "54"]
# 目标基因 ID
target_genes = ["ENSMUSG00000027951", "ENSMUSG00000020262", "ENSMUSG00000052551"] 

print("🚀 开始处理样本并保存矩阵到桌面...")

for s_id in sample_ids:
    h5ad_path = os.path.join(h5ad_dir, f"GSM81924{s_id}_GeneID_tissue.h5ad")
    
    if not os.path.exists(h5ad_path):
        print(f"⚠️ 找不到文件: {h5ad_path}")
        continue
    
    # 加载 H5AD
    adata = anndata.read_h5ad(h5ad_path)
    
    # 初始化 3x9 矩阵
    current_matrix = np.zeros((3, 6))
    
    # 遍历 6 个 Cluster
    for n in range(6):
        txt_path = os.path.join(cluster_dir, f"{s_id}_cluster{n}.txt")
        
        if os.path.exists(txt_path):
            with open(txt_path, 'r') as f:
                bin_indices = [line.strip() for line in f if line.strip()]
            
            # 过滤有效的 bin
            valid_bins = [b for b in bin_indices if b in adata.obs_names]
            
            if valid_bins:
                # 提取数据 (X 矩阵)
                sub_X = adata[valid_bins, target_genes].X
                
                # 处理稀疏矩阵并按 bin 求和
                if hasattr(sub_X, "toarray"):
                    gene_sums = np.array(sub_X.toarray().sum(axis=0)).flatten()
                else:
                    gene_sums = np.array(sub_X.sum(axis=0)).flatten()
                
                # 填充到第 n 列
                current_matrix[:, n] = gene_sums
        
    # 转为 DataFrame
    df = pd.DataFrame(
        current_matrix, 
        index=target_genes, 
        columns=[f"Cluster_{i}" for i in range(6)]
    )
    
    # 保存到指定目录
    save_path = os.path.join(save_dir, f"Sample_{s_id}_gene_matrix.csv")
    df.to_csv(save_path)
    
    print(f"✅ 样本 {s_id} 矩阵已保存至: {save_path}")
    
    # 释放内存
    adata.file.close()

print("\n🎉 所有样本处理完毕！请在桌面 gene_cluster_expression 文件夹查看结果。")

🚀 开始处理样本并保存矩阵到桌面...
✅ 样本 49 矩阵已保存至: C:\Users\28616\Desktop\gene_cluster_expression\Sample_49_gene_matrix.csv
✅ 样本 50 矩阵已保存至: C:\Users\28616\Desktop\gene_cluster_expression\Sample_50_gene_matrix.csv
✅ 样本 51 矩阵已保存至: C:\Users\28616\Desktop\gene_cluster_expression\Sample_51_gene_matrix.csv
✅ 样本 52 矩阵已保存至: C:\Users\28616\Desktop\gene_cluster_expression\Sample_52_gene_matrix.csv
✅ 样本 53 矩阵已保存至: C:\Users\28616\Desktop\gene_cluster_expression\Sample_53_gene_matrix.csv
✅ 样本 54 矩阵已保存至: C:\Users\28616\Desktop\gene_cluster_expression\Sample_54_gene_matrix.csv

🎉 所有样本处理完毕！请在桌面 gene_cluster_expression 文件夹查看结果。
